In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

In [ ]:
print(x_train.shape, y_train.shape)
print(x_test.shape, y_test.shape)

plt.figure(figsize=(3, 3))
plt.imshow(x_train[0], cmap='gray')
plt.title(f"Label: {y_train[0]}")
plt.axis('off')
plt.show()

### Data Dimensions Explanation
- **Height**: 28 pixels
- **Width**: 28 pixels
- **Channels**: 1 (Grayscale). Fashion-MNIST images are grayscale, containing a single intensity value for each pixel. For processing through standard Keras Conv2D layers, we must reshape them to include the channel dimension explicitly as `(28, 28, 1)`.

In [ ]:
sample_image = x_train[0].astype('float32') / 255.0
sample_image_input = np.expand_dims(np.expand_dims(sample_image, axis=-1), axis=0)
print(sample_image_input.shape)

In [ ]:
horizontal_kernel = np.array([[-1, -2, -1],
                              [0, 0, 0],
                              [1, 2, 1]], dtype='float32')

vertical_kernel = np.array([[-1, 0, 1],
                            [-2, 0, 2],
                            [-1, 0, 1]], dtype='float32')

sharpen_kernel = np.array([[0, -1, 0],
                           [-1, 5, -1],
                           [0, -1, 0]], dtype='float32')

outline_kernel = np.array([[-1, -1, -1],
                           [-1, 8, -1],
                           [-1, -1, -1]], dtype='float32')

kernels = np.stack([horizontal_kernel, vertical_kernel, sharpen_kernel, outline_kernel], axis=-1)
kernels = np.expand_dims(kernels, axis=-2)

conv_layer = layers.Conv2D(filters=4, kernel_size=(3, 3), use_bias=False, padding='same')
conv_layer.build(input_shape=(None, 28, 28, 1))
conv_layer.set_weights([kernels])

outputs = conv_layer(sample_image_input)
feature_maps = outputs[0].numpy()

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
titles = ['Horizontal Edges', 'Vertical Edges', 'Sharpen', 'Outline']
for i in range(4):
    axes[i].imshow(feature_maps[:, :, i], cmap='gray')
    axes[i].set_title(titles[i])
    axes[i].axis('off')
plt.show()

### Convolution Operation and Filters
- **Convolution Operation**: A mathematical operation where a kernel (filter) slides across the input image, calculating element-wise multiplication and summing the results to produce a feature map.
- **Filters/Kernels**: Small matrices of weights designed to detect specific features. For example, edge detection kernels highlight transitions in intensity (horizontal or vertical edges), while sharpening kernels enhance fine details.

In [ ]:
random_conv_layer = layers.Conv2D(filters=8, kernel_size=(3, 3), activation='relu', padding='same')
random_outputs = random_conv_layer(sample_image_input)
random_feature_maps = random_outputs[0].numpy()

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i in range(8):
    ax = axes[i // 4, i % 4]
    ax.imshow(random_feature_maps[:, :, i], cmap='viridis')
    ax.set_title(f"Feature Map {i+1}")
    ax.axis('off')
plt.show()

### How CNNs Capture Patterns
- **Edges and Outlines**: Early layers in a CNN focus on basic structural elements like horizontal, vertical, and diagonal edges where pixel intensity changes rapidly.
- **Textures and Shapes**: As layers process these edge maps, subsequent operations combine them to recognize textures, gradients, and shapes, forming hierarchical representations of the input.

In [ ]:
conv_s1_valid = layers.Conv2D(filters=1, kernel_size=(3, 3), strides=1, padding='valid')
conv_s1_same = layers.Conv2D(filters=1, kernel_size=(3, 3), strides=1, padding='same')
conv_s2_valid = layers.Conv2D(filters=1, kernel_size=(3, 3), strides=2, padding='valid')
conv_s2_same = layers.Conv2D(filters=1, kernel_size=(3, 3), strides=2, padding='same')

out_s1_valid = conv_s1_valid(sample_image_input)
out_s1_same = conv_s1_same(sample_image_input)
out_s2_valid = conv_s2_valid(sample_image_input)
out_s2_same = conv_s2_same(sample_image_input)

print("Input shape:", sample_image_input.shape)
print("Stride 1, Valid padding shape:", out_s1_valid.shape)
print("Stride 1, Same padding shape:", out_s1_same.shape)
print("Stride 2, Valid padding shape:", out_s2_valid.shape)
print("Stride 2, Same padding shape:", out_s2_same.shape)

### Stride and Padding
- **Stride**: The step size the kernel takes as it slides across the image. A larger stride (e.g., 2) reduces the spatial dimensions of the output feature map significantly by skipping pixels.
- **Padding**: Adding border pixels (usually zeros) around the input.
  - **Valid Padding (no padding)**: The filter is restricted to valid positions inside the image, resulting in a reduction of spatial size (e.g., from 28x28 to 26x26 with a 3x3 kernel).
  - **Same Padding**: Zero-padding is added so the output size matches the input size divided by the stride.

In [ ]:
max_pooling = layers.MaxPooling2D(pool_size=(2, 2))
avg_pooling = layers.AveragePooling2D(pool_size=(2, 2))

out_max = max_pooling(sample_image_input)
out_avg = avg_pooling(sample_image_input)

print("Input shape:", sample_image_input.shape)
print("MaxPooling output shape:", out_max.shape)
print("AveragePooling output shape:", out_avg.shape)

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].imshow(sample_image, cmap='gray')
axes[0].set_title("Original")
axes[0].axis('off')

axes[1].imshow(out_max[0, :, :, 0], cmap='gray')
axes[1].set_title("Max Pooling")
axes[1].axis('off')

axes[2].imshow(out_avg[0, :, :, 0], cmap='gray')
axes[2].set_title("Average Pooling")
axes[2].axis('off')

plt.show()

### Dimensionality Reduction in Pooling
- **MaxPooling**: Selects the maximum value in each pooling window. It retains the most prominent features (e.g., bright edges) and introduces translation invariance.
- **AveragePooling**: Computes the average of all values in the window, smoothing out features.
- **Dimensionality Reduction**: Both techniques reduce the spatial dimensions (height and width) of the feature maps, reducing the computational complexity (parameter count) and helping prevent overfitting.

In [ ]:
x_train_norm = np.expand_dims(x_train.astype('float32') / 255.0, axis=-1)
x_test_norm = np.expand_dims(x_test.astype('float32') / 255.0, axis=-1)

x_train_sub = x_train_norm[:2000]
y_train_sub = y_train[:2000]
x_val_sub = x_train_norm[2000:2500]
y_val_sub = y_train[2000:2500]

model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1), name='conv1'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

In [ ]:
history = model.fit(x_train_sub, y_train_sub, epochs=10, batch_size=64, validation_data=(x_val_sub, y_val_sub))

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test_norm[:500], y_test[:500])
print(f"Test Accuracy: {test_accuracy:.4f}")

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
first_conv_model = models.Model(inputs=model.inputs, outputs=model.get_layer('conv1').output)
first_conv_output = first_conv_model(sample_image_input)
learned_feature_maps = first_conv_output[0].numpy()

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
for i in range(32):
    ax = axes[i // 8, i % 8]
    ax.imshow(learned_feature_maps[:, :, i], cmap='viridis')
    ax.axis('off')
plt.suptitle('Learned Feature Maps from the First Conv2D Layer')
plt.show()